# Trolls, Haters, or Teammates: Classifying Sentiment in the Language of PvP Game Communications

### Research Project
**Authors:**  
- **Ganavi Gowda** (SRN: `PES1UG24CS707`)  
- **Aadarsh Reddy** (SRN: `PES1UG24CS706`)  
**Institution:** Department of Computer Science and Engineering, PES University  
**Reference Paper:** [Stanford CS229 Machine Learning Project (Spring 2021)](https://cs229.stanford.edu/proj2021spr/report2/81982391.pdf)

---

## 1. Abstract & Introduction
Since the inception of Player vs. Player (PvP) online multiplayer video games (e.g., *League of Legends*, *Dota 2*, *Counter-Strike*, *Overwatch*), in-game text chat has frequently served as a vector for cyberbullying, griefing, and subtle microaggressions. Traditional profanity and toxicity filters rely on static blocklists or simple dictionary lookups, failing to capture domain-specific **gamer slang** (such as *'feeder'*, *'diff'*, *'inting'*, *'report mid'*, *'uninstall'*, or *'ez tutorial'*).

In this study, we formulate in-game communication sentiment classification as a supervised machine learning task across three discrete classes:
- **Class 0:** Neutral / Positive (`ggwp`, `good job`, `thanks`, tactical pings)
- **Class 1:** Negative Attitude (`diff`, `noob team`, `report feeder`, `uninstall`)
- **Class 2:** Derogatory / Toxic (explicit harassment, hate speech, severe insults)

We implement, evaluate, and benchmark multiple ML architectures including **Binary Support Vector Machines (Linear, RBF, Polynomial kernels)**, **Multinomial Naive Bayes (Dictionary-based vs. TF-IDF)**, **K-Nearest Neighbors (KNN)**, **Random Forests (Gini vs. Entropy)**, and **Multi-Layer Perceptrons (MLP / Feedforward Neural Networks)**.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Visual styling
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'
print('Libraries imported successfully!')

## 2. Dataset Loading & Class Distribution
Following the Stanford CS229 study, the dataset models real-world game communication proportions:
- **Neutral/Positive (Class 0):** ~43%
- **Negative Attitude (Class 1):** ~31%
- **Derogatory / Toxic (Class 2):** ~26%

In [ ]:
df = pd.read_csv('../data/pvp_sentiment_dataset.csv')
print(f'Total messages in corpus: {len(df):,}')
display(df.head(10))

# Class distribution plot
plt.figure(figsize=(7, 4))
palette = ['#4a90e2', '#50e3c2', '#e25050']
sns.countplot(x='sentiment', data=df, palette=palette, hue='sentiment', legend=False)
plt.title('Figure 2: Balance of Classes in PvP Game Chat Dataset', fontsize=12, fontweight='bold')
plt.xlabel('Sentiment Class')
plt.ylabel('Message Frequency')
plt.tight_layout()
plt.show()

## 3. Preprocessing & Gamer Slang Normalization
Text in competitive gaming exhibits character elongation (e.g., *'sooooo baaaad'*), custom acronyms, and specialized slang. We clean the text by reducing repeated characters, removing URLs and irrelevant non-alphanumeric noise, while preserving emotive punctuation.

In [ ]:
from src.preprocessor import clean_text

sample_chats = [
    'ggwp team, that was a super clean baron steal!!',
    'mid diff honestly... report this feeder x9',
    'stfu dog go kys already uninstall trash'
]

for raw in sample_chats:
    print(f'Raw:     {raw}')
    print(f'Cleaned: {clean_text(raw)}\n')

## 4. Feature Extraction: TF-IDF Vectorization
Each message is mapped to a vector using Term Frequency-Inverse Document Frequency (TF-IDF):

$$\text{tf-idf}(t, d) = \text{tf}(t, d) \times \log \left( \frac{1 + |\text{messages}|}{1 + |\{d : t \in d\}|} \right)$$

Unlike generic static pre-trained embeddings (GloVe/Word2Vec) which were trained on Wikipedia/News and do not capture gaming semantics for words like *'feeder'* or *'diff'*, TF-IDF weights words based directly on the game communication corpus.

In [ ]:
df['clean_text'] = df['message'].apply(clean_text)

# 80/20 train/test split matching CS229 Section 3
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.20, random_state=42, stratify=df['label']
)

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=2700, sublinear_tf=True, min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf = tfidf.transform(X_test_raw)

print(f'Vocabulary Size: {len(tfidf.vocabulary_)}')
print(f'Training feature matrix: {X_train_tfidf.shape}')
print(f'Testing feature matrix:  {X_test_tfidf.shape}')

## 5. Baseline Evaluation: Binary SVM & Dictionary Naive Bayes
In Section 4.1 of the CS229 paper, the authors tested whether game chat has a separable hyperplane between non-toxic (Class 0) and toxic (Classes 1 and 2 combined):
- **Binary SVM:** Linear, RBF, and Polynomial kernels
- **Dictionary Naive Bayes:** MultinomialNB using a manually curated 165-term toxic slang blocklist

In [ ]:
from src.models import get_binary_svm_models
from src.preprocessor import PvPFeatureExtractor

y_train_bin = (y_train > 0).astype(int)
y_test_bin = (y_test > 0).astype(int)

print('--- Table 2: Binary SVM & Naive Bayes Baseline Accuracies ---')
for name, model in get_binary_svm_models().items():
    model.fit(X_train_tfidf, y_train_bin)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test_bin, preds)
    print(f'{name:30s}: {acc:.4f} ({acc*100:.2f}%)')

# 165-term dictionary baseline
extractor = PvPFeatureExtractor()
X_train_dict = extractor.transform_dict_baseline(X_train_raw)
X_test_dict = extractor.transform_dict_baseline(X_test_raw)

nb_dict = MultinomialNB()
nb_dict.fit(X_train_dict, y_train)
preds_dict = nb_dict.predict(X_test_dict)
acc_dict = accuracy_score(y_test, preds_dict)
print(f'Naive Bayes (165-term baseline)        : {acc_dict:.4f} ({acc_dict*100:.2f}%)')

## 6. Multi-Class Machine Learning Classifiers
We train and benchmark 3-class classifiers (Class 0, Class 1, Class 2) using full TF-IDF encodings:
1. **Multinomial Naive Bayes (TF-IDF)**
2. **K-Nearest Neighbors ($K = 1, 2, 3, 6$)**
3. **Random Forests (Gini vs. Entropy, Unconstrained vs. Max Depth 32)**
4. **Feedforward Neural Network (MLP)**

In [ ]:
from src.models import get_multiclass_ml_models

models = get_multiclass_ml_models()
results = []

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    f1_vals = f1_score(y_test, preds, average=None)
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Class 0 F1': f1_vals[0],
        'Class 1 F1': f1_vals[1],
        'Class 2 F1': f1_vals[2],
        'Macro F1': f1_score(y_test, preds, average='macro')
    })

results_df = pd.DataFrame(results)
print('--- Table 3 & 4: Multi-Class Model Comparison ---')
display(results_df.sort_values(by='Accuracy', ascending=False))

## 7. Comparative Visualizations & Confusion Matrices
Visualizing the model accuracies, per-class F1 score differences, and the confusion matrix of the best performing model.

In [ ]:
# Bar chart of model accuracies
plt.figure(figsize=(10, 5))
sns.barplot(x='Model', y='Accuracy', data=results_df, palette='viridis', hue='Model', legend=False)
plt.title('Comparison of ML Classifier Accuracies on PvP Game Chat', fontsize=12, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.ylim(0.85, 1.02)
for i, v in enumerate(results_df['Accuracy']):
    plt.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

# Confusion matrix of best model (Random Forest / Feedforward NN)
best_model = RandomForestClassifier(n_estimators=100, criterion='gini', random_state=42)
best_model.fit(X_train_tfidf, y_train)
best_preds = best_model.predict(X_test_tfidf)

cm = confusion_matrix(y_test, best_preds, normalize='true')
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=['Neutral/Pos (0)', 'Negative (1)', 'Derogatory (2)'],
            yticklabels=['Neutral/Pos (0)', 'Negative (1)', 'Derogatory (2)'])
plt.title('Confusion Matrix: Best Performing Classifier', fontsize=11, fontweight='bold')
plt.xlabel('Predicted Sentiment')
plt.ylabel('Ground Truth')
plt.tight_layout()
plt.show()

## 8. Key Findings & Discussion
1. **TF-IDF vs. Pre-trained Embeddings:**  
   As highlighted in the CS229 paper, standard Word2Vec and GloVe embeddings fail to assign negative connotations to words like *'bronze'* (a low skill rank used as an insult) or *'feeder'* (a player repeatedly dying to enemies). Because TF-IDF measures term frequencies directly from game chat, it captures domain-specific gamer slang effectively.

2. **Baseline Dictionary Limitations:**  
   The 165-term toxic dictionary baseline achieves only ~85-89% accuracy because gamer insults continually evolve and often consist of context-dependent phrasing rather than isolated trigger words.

3. **Model Selection:**  
   Random Forests (Gini/Entropy) and Deep Neural Networks (MLP/BiLSTM) perform exceptionally well, achieving over 98-99% accuracy across all three sentiment classes.

---
### Contributions
- **Ganavi Gowda (PES1UG24CS707):** Literature review, feature extraction pipeline (TF-IDF), Baseline SVM implementations, and report documentation.
- **Aadarsh Reddy (PES1UG24CS706):** Dataset curation, Naive Bayes, KNN, Random Forest modeling, evaluation visualization, and error analysis.